# 13 - Scoring
Run PageRank and centrality algorithms on the Neptune knowledge graph, compute expertise scores, and write them back as node properties.

## Step 1: Load Configuration

In [ ]:
import boto3
import json
import time
from pathlib import Path
from cleanup_utils import ResourceTracker

config = json.loads(Path("config.json").read_text())
REGION = config["REGION"]
PREFIX = config["PROJECT_PREFIX"]
GRAPH_ID = config["NEPTUNE_GRAPH_ID"]

if not GRAPH_ID:
    raise RuntimeError("NEPTUNE_GRAPH_ID not set. Run 02-neptune-graph.ipynb first.")

neptune = boto3.client("neptune-graph", region_name=REGION)
tracker = ResourceTracker()

print(f"\u2713 Config loaded (graph={GRAPH_ID})")

## Step 2: Define Query Helper

In [ ]:
def run_query(query, parameters=None):
    """Execute an openCypher query against Neptune Analytics."""
    kwargs = {
        "graphIdentifier": GRAPH_ID,
        "queryString": query,
        "language": "OPEN_CYPHER"
    }
    if parameters:
        kwargs["parameters"] = parameters
    response = neptune.execute_query(**kwargs)
    return json.loads(response["payload"].read())

print("\u2713 Query helper defined")

## Step 3: Compute Weighted Expertise Score

Score = sum of (relationship strength * number of connections per topic).

In [ ]:
# Compute expertise score based on relationship strengths and document count
expertise_query = """
MATCH (p:Person)
OPTIONAL MATCH (p)-[r:KNOWS_ABOUT]->(t:Topic)
OPTIONAL MATCH (p)-[:AUTHORED]->(d:Document)
WITH p, 
     coalesce(sum(r.strength), 0) AS topic_strength,
     count(DISTINCT t) AS topic_count,
     count(DISTINCT d) AS doc_count
WITH p, 
     topic_strength + (doc_count * 0.2) AS raw_score,
     topic_count,
     doc_count
SET p.expertise_score = raw_score
RETURN p.name AS name, p.title AS title,
       raw_score AS expertise_score,
       topic_count AS topics,
       doc_count AS documents
ORDER BY expertise_score DESC
"""

result = run_query(expertise_query)
print("Expertise Scores:")
print(f"{'Name':<20} {'Title':<30} {'Score':<8} {'Topics':<8} {'Docs':<8}")
print("-" * 74)
for row in result["results"]:
    print(f"{row['name']:<20} {row['title']:<30} {row['expertise_score']:<8.3f} {row['topics']:<8} {row['documents']:<8}")

print(f"\n\u2713 Expertise scores computed and stored as node properties")

## Step 4: Run PageRank Algorithm

Neptune Analytics provides built-in graph algorithms including PageRank.

In [ ]:
# Neptune Analytics PageRank using built-in algorithm
pagerank_query = """
CALL neptune.algo.pageRank(
    {edgeLabels: ['KNOWS_ABOUT', 'AUTHORED', 'RELATED_TO'],
     dampingFactor: 0.85,
     numOfIterations: 20,
     concurrency: 4}
)
YIELD node, score
WHERE 'Person' IN labels(node)
RETURN node.name AS name, node.title AS title, score
ORDER BY score DESC
"""

result = run_query(pagerank_query)
print("PageRank Scores (People):")
print(f"{'Name':<20} {'Title':<30} {'PageRank':<10}")
print("-" * 60)
for row in result["results"]:
    print(f"{row['name']:<20} {row['title']:<30} {row['score']:<10.6f}")

print(f"\n\u2713 PageRank computed")

## Step 5: Store PageRank as Node Property

In [ ]:
# Write PageRank scores back to Person nodes
store_pagerank_query = """
CALL neptune.algo.pageRank(
    {edgeLabels: ['KNOWS_ABOUT', 'AUTHORED', 'RELATED_TO'],
     dampingFactor: 0.85,
     numOfIterations: 20,
     concurrency: 4}
)
YIELD node, score
WHERE 'Person' IN labels(node)
SET node.pagerank = score
RETURN node.name AS name, score
ORDER BY score DESC
"""

result = run_query(store_pagerank_query)
print(f"\u2713 PageRank stored on {len(result['results'])} Person nodes")
for row in result["results"]:
    print(f"  {row['name']}: {row['score']:.6f}")

## Step 6: Compute Degree Centrality

Degree centrality measures how connected each person is.

In [ ]:
# Degree centrality - count of all relationships
centrality_query = """
MATCH (p:Person)
OPTIONAL MATCH (p)-[r]-()
WITH p, count(r) AS degree
SET p.centrality = toFloat(degree)
RETURN p.name AS name, degree AS centrality
ORDER BY centrality DESC
"""

result = run_query(centrality_query)
print("Degree Centrality:")
for row in result["results"]:
    print(f"  {row['name']}: {row['centrality']} connections")

print(f"\n\u2713 Centrality scores stored")

## Step 7: Compute Betweenness Centrality

In [ ]:
# Betweenness centrality using Neptune Analytics built-in algorithm
betweenness_query = """
CALL neptune.algo.betweennessCentrality(
    {edgeLabels: ['KNOWS_ABOUT', 'AUTHORED', 'RELATED_TO'],
     concurrency: 4}
)
YIELD node, score
WHERE 'Person' IN labels(node)
SET node.betweenness = score
RETURN node.name AS name, score
ORDER BY score DESC
"""

result = run_query(betweenness_query)
print("Betweenness Centrality:")
for row in result["results"]:
    print(f"  {row['name']}: {row['score']:.4f}")

print(f"\n\u2713 Betweenness centrality stored")

## Step 8: Compute Composite Score

Combine PageRank, centrality, and expertise strength into a final composite score.

In [ ]:
# Composite score: weighted combination of all metrics
composite_query = """
MATCH (p:Person)
WITH p,
     coalesce(p.pagerank, 0) AS pr,
     coalesce(p.centrality, 0) AS cent,
     coalesce(p.expertise_score, 0) AS exp,
     coalesce(p.betweenness, 0) AS betw
WITH p,
     (pr * 100 * 0.3) + (cent * 0.2) + (exp * 0.4) + (betw * 10 * 0.1) AS composite
SET p.composite_score = composite
RETURN p.name AS name, p.title AS title,
       composite AS composite_score,
       p.pagerank AS pagerank,
       p.centrality AS centrality,
       p.expertise_score AS expertise,
       p.betweenness AS betweenness
ORDER BY composite_score DESC
"""

result = run_query(composite_query)
print("Composite Scores (final ranking):")
print(f"{'Rank':<5} {'Name':<20} {'Composite':<12} {'PR':<10} {'Cent':<8} {'Exp':<8} {'Betw':<8}")
print("-" * 71)
for i, row in enumerate(result["results"], 1):
    print(f"{i:<5} {row['name']:<20} {row['composite_score']:<12.4f} {row['pagerank']:<10.6f} {row['centrality']:<8.1f} {row['expertise']:<8.3f} {row['betweenness']:<8.4f}")

print(f"\n\u2713 Composite scores computed and stored")

## Step 9: Topic-Level Leaderboards

In [ ]:
# Generate per-topic leaderboards
topics_query = """
MATCH (t:Topic)
RETURN t.name AS topic
ORDER BY topic
"""
topics = run_query(topics_query)

print("Per-Topic Leaderboards:\n")
for topic_row in topics["results"]:
    topic_name = topic_row["topic"]
    lb_query = """
    MATCH (p:Person)-[r:KNOWS_ABOUT]->(t:Topic {name: $topic})
    RETURN p.name AS name, r.strength AS strength, p.composite_score AS composite
    ORDER BY r.strength DESC
    """
    lb_result = run_query(lb_query, parameters={"topic": {"value": topic_name}})
    
    if lb_result["results"]:
        print(f"  {topic_name}:")
        for row in lb_result["results"]:
            print(f"    {row['name']} (strength: {row['strength']}, composite: {row['composite']:.3f})")
        print()

print("\u2713 Topic leaderboards generated")

## Step 10: Score History Tracking

In [ ]:
# Store score computation timestamp
import datetime

timestamp = datetime.datetime.now(datetime.timezone.utc).isoformat()

timestamp_query = """
MATCH (p:Person)
SET p.scores_updated_at = $timestamp
RETURN count(p) AS updated_count
"""
result = run_query(timestamp_query, parameters={"timestamp": {"value": timestamp}})
print(f"\u2713 Score timestamps updated: {result['results'][0]['updated_count']} nodes")
print(f"  Timestamp: {timestamp}")

## Step 11: Verify All Scores Stored

In [ ]:
# Final verification
verify_query = """
MATCH (p:Person)
RETURN p.name AS name,
       p.expertise_score IS NOT NULL AS has_expertise,
       p.pagerank IS NOT NULL AS has_pagerank,
       p.centrality IS NOT NULL AS has_centrality,
       p.betweenness IS NOT NULL AS has_betweenness,
       p.composite_score IS NOT NULL AS has_composite
ORDER BY p.name
"""
result = run_query(verify_query)

print("Score property coverage:")
print(f"{'Name':<20} {'Expertise':<12} {'PageRank':<12} {'Centrality':<12} {'Betweenness':<12} {'Composite':<12}")
print("-" * 80)
all_complete = True
for row in result["results"]:
    print(f"{row['name']:<20} {'\u2713' if row['has_expertise'] else '\u2717':<12} {'\u2713' if row['has_pagerank'] else '\u2717':<12} {'\u2713' if row['has_centrality'] else '\u2717':<12} {'\u2713' if row['has_betweenness'] else '\u2717':<12} {'\u2713' if row['has_composite'] else '\u2717':<12}")
    if not all([row['has_expertise'], row['has_pagerank'], row['has_centrality'], row['has_composite']]):
        all_complete = False

if all_complete:
    print("\n\u2713 All scores present on all Person nodes")
else:
    print("\n\u26a0\ufe0f Some scores missing - check graph data")

## Summary

- [x] Weighted expertise score computed (topic strength + document count)
- [x] PageRank algorithm executed via Neptune Analytics built-in
- [x] Degree centrality computed
- [x] Betweenness centrality computed
- [x] Composite score (weighted combination of all metrics)
- [x] All scores stored as Person node properties (not separate nodes)
- [x] Per-topic leaderboards generated
- [x] Score computation timestamps recorded

**Next:** [14-deploy-runtime.ipynb](14-deploy-runtime.ipynb) — Deploy agents to AgentCore Runtime